# TP3. Polynômes et fonctions

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp3-polynomes-fonctions/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : évaluation naïve

Écrivez `evaluer_naif(P, x)` qui calcule $P(x)$ en suivant la définition, avec une
puissance par terme.

**Résultat attendu :** `evaluer_naif([2, 3, 0, 5], 2)` vaut `48`, car
$2 + 6 + 40 = 48$.


In [ ]:
def evaluer_naif(P, x):
    resultat = 0
    for i in range(len(P)):
        resultat += P[i] * x**i
    return resultat


print(evaluer_naif([2, 3, 0, 5], 2))

## Exercice 2.1 : Horner

Écrivez `evaluer(P, x)` par le schéma de Horner, et vérifiez qu'elle donne le même
résultat que la version naïve sur plusieurs points.

**Résultat attendu :** les deux fonctions coïncident, et Horner est plus rapide sur un
polynôme de degré élevé.


In [ ]:
def evaluer(P, x):
    resultat = 0
    for coefficient in reversed(P):
        resultat = resultat * x + coefficient
    return resultat


P = [2, 3, 0, 5]
for x in (-2, 0, 1, 2):
    assert evaluer(P, x) == evaluer_naif(P, x)

print("Horner donne les mêmes résultats que la méthode naïve.")

## Exercice 2.2 : mesurer, et se méfier de ses prévisions

Construisez des polynômes dont tous les coefficients valent 1, de degrés 1 000, 5 000
et 20 000, et comparez le temps des deux méthodes en `x = 1.0001`. Avant de lancer la
mesure, écrivez votre prévision : de combien Horner devrait-il être plus rapide ?

**Résultat attendu :** un rapport d'environ 2, **constant** quand le degré augmente.
Ce n'est probablement pas ce que vous aviez prévu.


In [ ]:
import time


print("Prévision : Horner devrait être environ deux fois plus rapide ici.")
for degre in (1_000, 5_000, 20_000):
    P = [1] * (degre + 1)

    debut = time.perf_counter()
    evaluer_naif(P, 1.0001)
    temps_naif = time.perf_counter() - debut

    debut = time.perf_counter()
    evaluer(P, 1.0001)
    temps_horner = time.perf_counter() - debut

    print(f"degré {degre}: naïf {temps_naif:.4f} s, Horner {temps_horner:.4f} s, rapport {temps_naif / temps_horner:.1f}")

## Exercice 2.3 : le cas où l'écart explose vraiment

Refaites la mesure en évaluant en `x = 3`, un **entier**, sur des degrés 200, 400
et 800. Le rapport reste-t-il constant ?

**Résultat attendu :** cette fois le rapport grandit avec le degré, d'environ 4 à
environ 6.


In [ ]:
for degre in (200, 400, 800):
    P = [1] * (degre + 1)

    debut = time.perf_counter()
    evaluer_naif(P, 3)
    temps_naif = time.perf_counter() - debut

    debut = time.perf_counter()
    evaluer(P, 3)
    temps_horner = time.perf_counter() - debut

    print(f"degré {degre}: naïf {temps_naif:.4f} s, Horner {temps_horner:.4f} s, rapport {temps_naif / temps_horner:.1f}")

## Exercice 3.1 : la dérivée

Écrivez `deriver(P)` qui renvoie la liste des coefficients du polynôme dérivé.

**Résultat attendu :** `deriver([2, 3, 0, 5])` vaut `[3, 0, 15]`, c'est-à-dire
$3 + 15x^{2}$.


In [ ]:
def deriver(P):
    return [i * P[i] for i in range(1, len(P))]


print(deriver([2, 3, 0, 5]))

## Exercice 4.1 : dichotomie

Écrivez `dichotomie(f, a, b, tolerance)` qui renvoie une racine de `f` entre `a` et
`b`. Appliquez-la à $x^{2} - 2$ sur $[0, 2]$.

**Résultat attendu :** une valeur proche de $\sqrt{2} \approx 1{,}414214$.


In [ ]:
def dichotomie(f, a, b, tolerance=1e-12):
    if f(a) == 0:
        return a
    if f(b) == 0:
        return b
    if f(a) * f(b) > 0:
        raise ValueError("f(a) et f(b) doivent être de signes opposés")

    while b - a > tolerance:
        milieu = (a + b) / 2
        if f(milieu) == 0:
            return milieu
        if f(a) * f(milieu) < 0:
            b = milieu
        else:
            a = milieu

    return (a + b) / 2


print(dichotomie(lambda x: x**2 - 2, 0, 2))

## Exercice 4.2 : Newton sur un polynôme

Écrivez `newton(P, depart, tolerance)` qui cherche une racine du polynôme `P` en
utilisant `evaluer` et `deriver`. Appliquez-la à $x^{3} - 2x - 5$, en partant de 2.

**Résultat attendu :** environ `2.0945515`, atteint en cinq tours ou moins.


In [ ]:
def newton(P, depart, tolerance=1e-12):
    derivee = deriver(P)
    x = depart

    for _ in range(100):
        pente = evaluer(derivee, x)
        if pente == 0:
            raise ValueError("La dérivée est nulle")

        suivant = x - evaluer(P, x) / pente
        if abs(suivant - x) < tolerance:
            return suivant
        x = suivant

    raise ValueError("La méthode ne converge pas")


print(newton([-5, -2, 0, 1], 2))

## Exercice 4.3 : faire échouer Newton

Appliquez `newton` au polynôme $x^{3} - 2x + 2$, c'est-à-dire `[2, -2, 0, 1]`, en
partant de `0`. Affichez le trajet des premiers itérés pour comprendre ce qui se passe.

**Résultat attendu :** aucune convergence, et un trajet qui alterne entre `0` et `1`
indéfiniment.


In [ ]:
P = [2, -2, 0, 1]
derivee = deriver(P)
x = 0
trajet = [x]

for _ in range(7):
    x = x - evaluer(P, x) / evaluer(derivee, x)
    trajet.append(x)

print("Itérés de Newton :", trajet)

## Exercice 4.4 : le prix d'un mauvais point de départ

Reprenez $x^{3} - 2x - 5$ et comparez le nombre de tours de `newton` en partant
de `2`, puis de `0`.

**Résultat attendu :** cinq tours depuis `2`, une vingtaine depuis `0`.


In [ ]:
def compter_tours_newton(P, depart, tolerance=1e-12):
    derivee = deriver(P)
    x = depart

    for tours in range(1, 101):
        suivant = x - evaluer(P, x) / evaluer(derivee, x)
        if abs(suivant - x) < tolerance:
            return suivant, tours
        x = suivant

    return x, tours


P = [-5, -2, 0, 1]
for depart in (2, 0):
    racine, tours = compter_tours_newton(P, depart)
    print(f"Départ {depart} : {tours} tours, racine {racine}")

## Exercice 5.1 : lire le graphique

Sur le tracé, repérez la racine trouvée par Newton. Combien de fois la courbe
coupe-t-elle l'axe des abscisses ? Ce polynôme est de degré 3 : que dire de ses autres
racines ?


In [ ]:
racine = newton([-5, -2, 0, 1], 2)
print(f"La courbe coupe l'axe des abscisses une seule fois, vers x = {racine:.7f}.")
print("Comme le polynôme est de degré 3, ses deux autres racines sont complexes conjugées.")

## Exercice 6.1 : somme et produit

Écrivez `somme(P, Q)` et `produit_poly(P, Q)`. Attention, les deux listes n'ont pas
forcément la même longueur.

**Résultat attendu :** `produit_poly([1, 1], [1, 1])` vaut `[1, 2, 1]`, c'est-à-dire
$(1 + x)^{2} = 1 + 2x + x^{2}$, et `produit_poly([-1, 1], [1, 1])` vaut
`[-1, 0, 1]`, l'identité $(x - 1)(x + 1) = x^{2} - 1$.


In [ ]:
def somme(P, Q):
    longueur = max(len(P), len(Q))
    return [
        (P[i] if i < len(P) else 0) + (Q[i] if i < len(Q) else 0)
        for i in range(longueur)
    ]


def produit_poly(P, Q):
    if not P or not Q:
        return []

    resultat = [0] * (len(P) + len(Q) - 1)
    for i in range(len(P)):
        for j in range(len(Q)):
            resultat[i + j] += P[i] * Q[j]
    return resultat


print(somme([1, 2], [3, 4, 5]))
print(produit_poly([1, 1], [1, 1]))
print(produit_poly([-1, 1], [1, 1]))

## Exercice 6.2 : le triangle de Pascal, par les polynômes

Calculez $(1 + x)^{10}$ en multipliant dix fois `[1, 1]` par lui-même. Que
reconnaissez-vous dans la liste obtenue ?

**Résultat attendu :** `[1, 10, 45, 120, 210, 252, 210, 120, 45, 10, 1]`.


In [ ]:
resultat = [1, 1]
for _ in range(9):
    resultat = produit_poly(resultat, [1, 1])

print(resultat)
print("Ce sont les coefficients binomiaux de la ligne 10 du triangle de Pascal.")

## Exercice 7.1 : trouver le meilleur pas

Écrivez `derivee_avant(f, x, h)` et `derivee_centree(f, x, h)`. Calculez l'erreur de
chacune sur $\sin$ en $x = 1$, dont la dérivée exacte est $\cos 1$, pour
`h` valant $10^{-1}, 10^{-2}, 10^{-4}, 10^{-6}, 10^{-8}, 10^{-10}, 10^{-12}$.

**Résultat attendu :** l'erreur diminue puis **remonte**. Le meilleur pas est de
l'ordre de $10^{-8}$ pour la formule avant, $10^{-6}$ pour la formule centrée,
et à $h = 10^{-12}$ l'erreur est remontée de quatre à six ordres de grandeur
par rapport à ce meilleur pas.


In [ ]:
import math


def derivee_avant(f, x, h):
    return (f(x + h) - f(x)) / h


def derivee_centree(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)


for h in (1e-1, 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12):
    erreur_avant = abs(derivee_avant(math.sin, 1, h) - math.cos(1))
    erreur_centree = abs(derivee_centree(math.sin, 1, h) - math.cos(1))
    print(f"h={h:.0e} : avant={erreur_avant:.3e}, centrée={erreur_centree:.3e}")

## Exercice 8.1 : les trois méthodes

Écrivez `rectangles(f, a, b, n)`, `trapezes(f, a, b, n)` et `simpson(f, a, b, n)`.
Testez-les sur $\int_0^1 x^{2}\,dx = 1/3$ pour `n` valant 10, 100 et 1000.

**Résultat attendu :** l'erreur des rectangles est divisée par 10 quand `n` est
multiplié par 10, celle des trapèzes par 100, et Simpson donne le résultat exact, à
un arrondi de $10^{-17}$ près.


In [ ]:
def rectangles(f, a, b, n):
    h = (b - a) / n
    return h * sum(f(a + i * h) for i in range(n))


def trapezes(f, a, b, n):
    h = (b - a) / n
    resultat = (f(a) + f(b)) / 2
    for i in range(1, n):
        resultat += f(a + i * h)
    return h * resultat


def simpson(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("n doit être pair")

    h = (b - a) / n
    resultat = f(a) + f(b)
    for i in range(1, n):
        coefficient = 4 if i % 2 == 1 else 2
        resultat += coefficient * f(a + i * h)
    return h * resultat / 3


for n in (10, 100, 1000):
    print(
        n,
        rectangles(lambda x: x**2, 0, 1, n),
        trapezes(lambda x: x**2, 0, 1, n),
        simpson(lambda x: x**2, 0, 1, n),
    )

## Exercice 8.2 : calculer π

On a $\int_0^1 \frac{4}{1 + x^{2}}\,dx = \pi$. Calculez cette intégrale par les
trapèzes et par Simpson avec `n = 1000`, et comparez à `math.pi`.

**Résultat attendu :** une erreur de l'ordre de $10^{-7}$ pour les trapèzes, et
un résultat indistinguable de `math.pi` pour Simpson.


In [ ]:
import math


f = lambda x: 4 / (1 + x**2)
pi_trapezes = trapezes(f, 0, 1, 1000)
pi_simpson = simpson(f, 0, 1, 1000)

print(f"Trapèzes : {pi_trapezes}, erreur : {abs(pi_trapezes - math.pi):.3e}")
print(f"Simpson : {pi_simpson}, erreur : {abs(pi_simpson - math.pi):.3e}")